# Phase 1 Baseline Validation

Run 5 experiments to establish baseline statistics for Phase 2 meta-learning:
- **3-seed baseline**: AdamW, lr=0.001, seeds=[42, 123, 456]
- **Higher LR variant**: lr=0.003, seed=42
- **SGD baseline**: SGD, lr=0.01, momentum=0.9, seed=42

Includes test set evaluation, timing metrics, and saves `phase1_baseline_results.json`.

In [ ]:
# Setup and imports
import sys
import json
import time
from pathlib import Path

candidate_roots = [
    Path('/workspaces/ouroboros'),
    Path('/content/drive/MyDrive/ouroboros'),
    Path('/content/ouroboros'),
    Path('.').resolve().parent,  # Local development
]
project_root = next((p for p in candidate_roots if (p / 'src').exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
from torch import nn
from torch.optim import AdamW, SGD
from tqdm.auto import tqdm

from src.data_loaders import (
    get_cifar10_loaders, get_cifar10_test_loader,
)
from src.models import CNN3Layer, DEFAULT_CHANNELS, count_parameters
from src.trainer import train_epoch, validate_epoch, get_epoch_scheduler
from src.utils import set_seed, get_device, ensure_dirs

device = get_device()
ensure_dirs('results/phase1', 'checkpoints')
print(f"Device: {device}")
print(f"Project root: {project_root}")

Loaded train_reproducibility from: /content/drive/MyDrive/ouroboros/src/train_reproducibility.py


CIFAR-100 seed=1337: 100%|██████████| 5/5 [01:50<00:00, 22.14s/epoch]


Deterministic status: not_run
Summary:
{'dataset': 'CIFAR-10', 'mean_accuracy': 0.6033333333333334, 'std_accuracy': 0.0049270229911738215, 'min_accuracy': 0.5964, 'max_accuracy': 0.6074, 'target_accuracy': 0.65}
{'dataset': 'Fashion-MNIST', 'mean_accuracy': 0.8450666666666667, 'std_accuracy': 0.004502098276236194, 'min_accuracy': 0.8387, 'max_accuracy': 0.8483, 'target_accuracy': 0.88}
{'dataset': 'CIFAR-100', 'mean_accuracy': 0.26676666666666665, 'std_accuracy': 0.0034179265969623046, 'min_accuracy': 0.2623, 'max_accuracy': 0.2706, 'target_accuracy': 0.4}


## Define Experiment Runner

In [ ]:
def run_experiment(config: dict) -> dict:
    """Run a single experiment with config dict. Returns results with test accuracy."""
    set_seed(config["seed"], deterministic=False)
    
    train_loader, val_loader = get_cifar10_loaders(config["batch_size"], config["num_workers"], config["data_dir"], seed=config["seed"])
    test_loader = get_cifar10_test_loader(config["batch_size"], config["num_workers"], config["data_dir"])
    
    model = CNN3Layer(num_classes=10, in_channels=3, channels=DEFAULT_CHANNELS).to(device)
    
    if config["optimizer"] == "sgd":
        optimizer = SGD(model.parameters(), lr=config["lr"], momentum=config.get("momentum", 0.9), weight_decay=config["weight_decay"])
    else:
        optimizer = AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    
    criterion = nn.CrossEntropyLoss()
    scheduler = get_epoch_scheduler(optimizer, config["epochs"], config["warmup_epochs"])
    
    print(f"[OPTIMIZER] {type(optimizer).__name__} | lr={config['lr']} | wd={config['weight_decay']}")
    print(f"[MODEL] params={count_parameters(model):,}")
    
    # NVIDIA Best Practice: Warm-up iteration
    if device.type == "cuda":
        warmup_batch = next(iter(train_loader))
        with torch.no_grad():
            _ = model(warmup_batch[0].to(device))
        torch.cuda.synchronize()
    
    epoch_times, train_history, val_history = [], [], []
    
    # NVIDIA Best Practice: GPU sync for accurate timing
    if device.type == "cuda":
        torch.cuda.synchronize()
    total_start = time.perf_counter()
    
    for epoch in tqdm(range(1, config["epochs"] + 1), desc=config["name"], unit="epoch"):
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_start = time.perf_counter()
        
        train_metrics = train_epoch(model, train_loader, optimizer, criterion, device, amp_enabled=(device.type == "cuda"))
        val_metrics = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()
        
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - epoch_start)
        train_history.append(train_metrics)
        val_history.append(val_metrics)
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    total_time = time.perf_counter() - total_start
    
    test_metrics = validate_epoch(model, test_loader, criterion, device)
    
    # Memory cleanup
    if device.type == "cuda":
        torch.cuda.empty_cache()
    
    return {
        "experiment_name": config["name"],
        "config": {k: v for k, v in config.items() if k != "name"},
        "final_train_acc": train_history[-1]["accuracy"],
        "final_val_acc": val_history[-1]["accuracy"],
        "test_acc": test_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "total_time_sec": total_time,
        "mean_epoch_time": sum(epoch_times) / len(epoch_times),
        "params": count_parameters(model),
    }

## Run 5 Experiments

In [ ]:
# Default config
DEFAULT_CONFIG = {
    "epochs": 5,
    "batch_size": 128,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "warmup_epochs": 1,
    "optimizer": "adamw",
    "momentum": 0.9,
    "num_workers": 2,
    "data_dir": "assets",
}

# 5 experiments: 3-seed baseline + 2 variants
experiments = [
    {"name": "baseline_seed42", **DEFAULT_CONFIG, "seed": 42},
    {"name": "baseline_seed123", **DEFAULT_CONFIG, "seed": 123},
    {"name": "baseline_seed456", **DEFAULT_CONFIG, "seed": 456},
    {"name": "higher_lr", **DEFAULT_CONFIG, "seed": 42, "lr": 0.003},
    {"name": "sgd_baseline", **DEFAULT_CONFIG, "seed": 42, "optimizer": "sgd", "lr": 0.01},
]

# Run all experiments
all_results = []
for exp in experiments:
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp['name']}")
    print(f"{'='*70}")
    
    result = run_experiment(exp)
    all_results.append(result)
    
    print(f"\n[RESULTS]")
    print(f"  Train Acc: {result['final_train_acc']:.4f} ({result['final_train_acc']*100:.2f}%)")
    print(f"  Val Acc:   {result['final_val_acc']:.4f} ({result['final_val_acc']*100:.2f}%)")
    print(f"  Test Acc:  {result['test_acc']:.4f} ({result['test_acc']*100:.2f}%)")
    print(f"  Test Loss: {result['test_loss']:.4f}")
    print(f"  Total Time: {result['total_time_sec']:.1f}s")
    print(f"  Avg Epoch Time: {result['mean_epoch_time']:.2f}s")


## Compute Baseline Statistics & Save Results

In [ ]:
# Compute baseline statistics (first 3 experiments = 3-seed baseline)
baseline_results = all_results[:3]
baseline_val_accs = [r["final_val_acc"] for r in baseline_results]
baseline_test_accs = [r["test_acc"] for r in baseline_results]

def mean(vals): return sum(vals) / len(vals)
def std(vals): 
    m = mean(vals)
    return (sum((x - m)**2 for x in vals) / len(vals)) ** 0.5

stats = {
    "baseline_val_mean": mean(baseline_val_accs),
    "baseline_val_std": std(baseline_val_accs),
    "baseline_test_mean": mean(baseline_test_accs),
    "baseline_test_std": std(baseline_test_accs),
}

# Save results
output = {
    "experiments": all_results,
    "baseline_statistics": stats,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}

output_path = Path("results/phase1/phase1_baseline_results.json")
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"\n{'='*70}")
print("PHASE 1 BASELINE STATISTICS (3-SEED AVERAGE)")
print(f"{'='*70}")
print(f"Seeds: {[r['config']['seed'] for r in baseline_results]}")
print(f"\nValidation Accuracy: {stats['baseline_val_mean']:.4f} ± {stats['baseline_val_std']:.4f}")
print(f"                     ({stats['baseline_val_mean']*100:.2f}% ± {stats['baseline_val_std']*100:.2f}%)")
print(f"\nTest Accuracy:       {stats['baseline_test_mean']:.4f} ± {stats['baseline_test_std']:.4f}")
print(f"                     ({stats['baseline_test_mean']*100:.2f}% ± {stats['baseline_test_std']*100:.2f}%)")
print(f"\nResults saved to: {output_path}")
print(f"{'='*70}")


## Results Summary Table

In [ ]:
# Display results table
print(f"\n{'='*70}")
print("DETAILED RESULTS TABLE")
print(f"{'='*70}")
print(f"{'Experiment':<20} {'Optimizer':<8} {'LR':<8} {'Val Acc':<10} {'Test Acc':<10} {'Time (s)':<10}")
print("-" * 70)
for r in all_results:
    cfg = r["config"]
    print(f"{r['experiment_name']:<20} {cfg['optimizer']:<8} {cfg['lr']:<8.4f} {r['final_val_acc']:<10.4f} {r['test_acc']:<10.4f} {r['total_time_sec']:<10.1f}")
print("=" * 70)


In [ ]:
# Final Summary
print(f"\n{'='*70}")
print("FINAL SUMMARY - Phase 1 Reproducibility Study")
print(f"{'='*70}")
print(f"Dataset: CIFAR-10")
print(f"Model Architecture: CNN3Layer")
print(f"Model Parameters: {all_results[0]['params']:,}")
print(f"Training Epochs: {experiments[0]['epochs']}")
print(f"Batch Size: {experiments[0]['batch_size']}")
print(f"\nTotal Experiments Run: {len(all_results)}")
print(f"  - Baseline (3 seeds): {len(baseline_results)}")
print(f"  - Variants: {len(all_results) - len(baseline_results)}")

print(f"\n{'─'*70}")
print("BASELINE PERFORMANCE (AdamW, lr=0.001, 3 seeds)")
print(f"{'─'*70}")
for i, r in enumerate(baseline_results, 1):
    print(f"  Seed {r['config']['seed']:>3}: Val={r['final_val_acc']:.4f} | Test={r['test_acc']:.4f}")
print(f"\n  Average:  Val={stats['baseline_val_mean']:.4f}±{stats['baseline_val_std']:.4f} | Test={stats['baseline_test_mean']:.4f}±{stats['baseline_test_std']:.4f}")

print(f"\n{'─'*70}")
print("VARIANT EXPERIMENTS")
print(f"{'─'*70}")
for r in all_results[3:]:
    cfg = r["config"]
    print(f"  {r['experiment_name']:<20}: {cfg['optimizer']}, lr={cfg['lr']:.4f}")
    print(f"    Val={r['final_val_acc']:.4f} | Test={r['test_acc']:.4f} | Time={r['total_time_sec']:.1f}s")

# Find best and worst performers
best_test = max(all_results, key=lambda x: x['test_acc'])
worst_test = min(all_results, key=lambda x: x['test_acc'])
fastest = min(all_results, key=lambda x: x['total_time_sec'])

print(f"\n{'─'*70}")
print("KEY FINDINGS")
print(f"{'─'*70}")
print(f"  Best Test Accuracy:  {best_test['experiment_name']} ({best_test['test_acc']:.4f})")
print(f"  Worst Test Accuracy: {worst_test['experiment_name']} ({worst_test['test_acc']:.4f})")
print(f"  Fastest Training:    {fastest['experiment_name']} ({fastest['total_time_sec']:.1f}s)")
print(f"\n  Reproducibility (3-seed std): ±{stats['baseline_test_std']:.4f} ({stats['baseline_test_std']*100:.2f}%)")

print(f"\n{'='*70}")
print(f"Phase 1 Complete! Results saved to: {output_path}")
print(f"{'='*70}")


## Final Comprehensive Summary